# Fragment Ion Intensity — Inference with Model Features

Predicting fragment ion intensities with a `PrositIntensityPredictor`. This model takes
**metadata features alongside the sequence** (`precursor_charge_onehot`,
`collision_energy_aligned_normed`), so those must be supplied at inference time too.

In [1]:
import numpy as np
import pandas as pd
from datasets import Dataset

from dlomix.data import FragmentIonIntensityDataset
from dlomix.models import PrositIntensityPredictor
from dlomix.losses import masked_spectral_distance
from dlomix.pipelines import InferencePipeline

Using TensorFlow Backend for DLOmix. To change the backend, set the DLOMIX_BACKEND environment variable to tensorflow or pytorch and re-import DLOmix.

Available feature extractors are (use the key of the following dict and pass it to features_to_extract in the Dataset Class):
{
   "atom_count": "Atom count of PTM.",
   "delta_mass": "Delta mass of PTM.",
   "mod_gain": "Gain of atoms due to PTM.",
   "mod_loss": "Loss of atoms due to PTM.",
   "red_smiles": "Reduced SMILES representation of PTM."
}.
When writing your own feature extractor, you can either
    (1) use the FeatureExtractor class or
    (2) write a function that can be mapped to the Hugging Face dataset.
In both cases, you can access the parsed sequence information from the dataset using the following keys, which all provide python lists:
    - _parsed_sequence: parsed sequence
    - _n_term_mods: N-terminal modifications
    - _c_term_mods: C-terminal modifications



## 1. Data + a trained model (mocked)

Replace with your data and real training. Here we use the bundled sample and a 1-epoch fit.

In [2]:
from pathlib import Path

# the small sample shipped with the repo (run from repo root or notebooks/)
DATA = next(p for p in [
    'example_dataset/intensity/third_pool_processed_sample.parquet',
    '../example_dataset/intensity/third_pool_processed_sample.parquet',
] if Path(p).exists())

df = pd.read_parquet(DATA).head(256)   # subset for a fast example
df[['modified_sequence', 'precursor_charge_onehot', 'collision_energy_aligned_normed']].head(3)

,modified_sequence,precursor_charge_onehot,collision_energy_aligned_normed
0,EPPVIC[UNIMOD:4]SHFAQDLWPEQGREDSFQK,"[0, 0, 1, 0, 0, 0]",0.253658
1,M[UNIMOD:35]VGNVTGAQAYASTTK,"[0, 1, 0, 0, 0, 0]",0.225589
2,NMMAAC[UNIMOD:4]DPRHGC[UNIMOD:4]YLTVAAIFR,"[0, 0, 0, 1, 0, 0]",0.322733


In [3]:
dataset = FragmentIonIntensityDataset(
    data_source=Dataset.from_pandas(df, preserve_index=False),
    data_format='hf',
    sequence_column='modified_sequence',
    label_column='intensities_raw',
    model_features=['precursor_charge_onehot', 'collision_energy_aligned_normed'],
    max_seq_len=30,
    batch_size=16,
    val_ratio=0.2,
)

model = PrositIntensityPredictor(
    seq_length=30,
    input_keys={'SEQUENCE_KEY': 'modified_sequence'},
    meta_data_keys={
        'COLLISION_ENERGY_KEY': 'collision_energy_aligned_normed',
        'PRECURSOR_CHARGE_KEY': 'precursor_charge_onehot',
    },
    use_meta_data=True,
    alphabet=dataset.extended_alphabet,
)
model.compile(optimizer='adam', loss=masked_spectral_distance)
model.fit(dataset.tensor_train_data, epochs=1, verbose=0)  # mock training

## 2. Inference on raw inputs

Because the model uses metadata, the inputs are a **dict**: sequences plus the same feature
columns. The preprocessor handles encoding/padding and assembles the model's tensor dict.

In [4]:
raw_inputs = {
    'modified_sequence': df['modified_sequence'].tolist()[:5],
    'precursor_charge_onehot': [list(x) for x in df['precursor_charge_onehot'][:5]],
    'collision_energy_aligned_normed': df['collision_energy_aligned_normed'].tolist()[:5],
}

pipeline = InferencePipeline.from_model_and_dataset(model, dataset)
predictions = pipeline.predict(raw_inputs)
predictions.shape   # (n_peptides, n_fragment_ions)

1/1 [==============================] - 1s 1s/step


(5, 174)

## 3. Save / load locally


In [5]:
pipeline.save('artifacts/intensity_model', overwrite=True)
reloaded = InferencePipeline.load('artifacts/intensity_model')
reloaded.predict(raw_inputs).shape

1/1 [==============================] - 1s 791ms/step


(5, 174)

## 4. Share on the HuggingFace Hub

The bundle (model + preprocessor + metadata) is pushed as one repo and loaded back with
`from_pretrained` — no training or dataset needed.

In [6]:
# requires HF auth once: `huggingface-cli login`
pipeline.push_to_hub('omsh/test-model', private=True)

reloaded = InferencePipeline.from_pretrained('omsh/test-model')
reloaded.predict(raw_inputs).shape

1/1 [==============================] - 1s 667ms/step


(5, 174)

**Notes**

- Metadata models need their `model_features` provided at inference — missing columns raise a
  clear error.
- This pushes to the shared `omsh/test-model` repo and **overwrites** its contents.